# Задание 1. Анализ экспорта и импорта важнейших видов продукции Республики Беларусь (2000–2021)
Источник: Белстат. Файлы `export_tovarov_vazh-2021-1.xlsx` и `import_tovarov_vazh-2021-1.xlsx`.

In [ ]:
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

In [ ]:
# загрузка файлов в Colab
from google.colab import files
files.upload()

## 1. Загрузка и очистка данных

In [ ]:
def load(path):
    df = pd.read_excel(path, header=3, index_col=0).dropna(how="all")
    df.index = df.index.str.replace(r"\s+", " ", regex=True).str.strip()
    df.columns = df.columns.astype(int)
    # числа вида "1 196,8" приходят строками
    df = df.astype(str).apply(lambda c: c.str.replace(r"\s", "", regex=True).str.replace(",", "."))
    return df.astype(float)


def split_units(df):
    # "Сахар, тыс. т" -> название "Сахар", единица "тыс. т"
    names = [i.rsplit(", ", 1)[0] for i in df.index]
    units = [i.rsplit(", ", 1)[1] for i in df.index]
    df = df.copy()
    df.index = names
    return df, pd.Series(units, index=names)


exp, exp_units = split_units(load("export_tovarov_vazh-2021-1.xlsx"))
imp, imp_units = split_units(load("import_tovarov_vazh-2021-1.xlsx"))

print("Экспорт:", exp.shape, "| Импорт:", imp.shape)
print("Пропусков:", exp.isna().sum().sum(), imp.isna().sum().sum())
exp.head()

In [ ]:
imp.head()

Товары измеряются в разных единицах (тонны, штуки, м³, млн долл.), поэтому складывать их нельзя.
Дальше сравниваем динамику, темпы роста и относительные значения.

## 2. Динамика по каждому товару

In [ ]:
def plot_grid(df, units, title):
    n = len(df)
    rows = int(np.ceil(n / 4))
    fig, axes = plt.subplots(rows, 4, figsize=(18, 3.3 * rows))
    for ax, name in zip(axes.flat, df.index):
        ax.plot(df.columns, df.loc[name], marker="o", ms=3)
        ax.set_title(textwrap.fill(name, 32), fontsize=9)
        ax.set_ylabel(units[name], fontsize=8)
        ax.tick_params(labelsize=8)
    for ax in axes.flat[n:]:
        ax.axis("off")
    fig.suptitle(title, fontsize=15)
    plt.tight_layout()
    plt.show()


plot_grid(exp, exp_units, "Экспорт важнейших видов продукции, 2000–2021")

In [ ]:
plot_grid(imp, imp_units, "Импорт важнейших видов продукции, 2000–2021")

## 3. Когда каждый товар был на пике

In [ ]:
def plot_heatmap(df, title):
    share = df.div(df.max(axis=1), axis=0)     # значение / максимум по товару
    plt.figure(figsize=(14, 0.45 * len(df) + 2))
    sns.heatmap(share, cmap="YlGnBu", cbar_kws={"label": "доля от максимума"})
    plt.title(title)
    plt.ylabel("")
    plt.show()


plot_heatmap(exp, "Экспорт: значение относительно максимума за период")
plot_heatmap(imp, "Импорт: значение относительно максимума за период")

## 4. Темпы роста 2000–2021

In [ ]:
def growth_table(df):
    t = pd.DataFrame({
        "2000": df[2000],
        "2021": df[2021],
        "рост, раз": df[2021] / df[2000],
        "CAGR, %": ((df[2021] / df[2000]) ** (1 / 21) - 1) * 100,
        "год максимума": df.idxmax(axis=1),
    })
    return t.sort_values("CAGR, %", ascending=False).round(2)


exp_growth = growth_table(exp)
imp_growth = growth_table(imp)
exp_growth

In [ ]:
imp_growth

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, g, title in zip(axes, [exp_growth, imp_growth], ["Экспорт", "Импорт"]):
    g["CAGR, %"].sort_values().plot.barh(ax=ax, color="steelblue")
    ax.set_title(f"{title}: среднегодовой темп роста 2000–2021, %")
    ax.set_ylabel("")
    ax.axvline(0, color="black", lw=0.8)
plt.tight_layout()
plt.show()

CAGR у легковых автомобилей (экспорт) и машин специального назначения (импорт) завышен — стартовая база в 2000 году очень маленькая.

## 5. Экспорт и импорт одних и тех же товаров

In [ ]:
# у легковых авто в экспорте штуки, в импорте тыс. шт. - приводим к тыс. шт.
exp.loc["Легковые автомобили"] = exp.loc["Легковые автомобили"] / 1000
exp_units["Легковые автомобили"] = "тыс. шт."

common = exp.index.intersection(imp.index)
print(list(common))

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
for ax, name in zip(axes.flat, common):
    ax.plot(exp.columns, exp.loc[name], label="экспорт", marker="o", ms=3)
    ax.plot(imp.columns, imp.loc[name], label="импорт", marker="o", ms=3)
    ax.set_title(textwrap.fill(name, 45))
    ax.set_ylabel(exp_units[name])
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# соотношение экспорт / импорт и связь динамики
rows = []
for name in common:
    ratio_2021 = exp.loc[name, 2021] / imp.loc[name, 2021]
    corr = exp.loc[name].pct_change().corr(imp.loc[name].pct_change(), method="spearman")
    rows.append([name, round(ratio_2021, 2), round(corr, 2)])

pd.DataFrame(rows, columns=["товар", "экспорт/импорт в 2021", "корреляция годовых изменений"])

## 6. Годовые изменения и кризисные годы

In [ ]:
# доля товаров, у которых в этом году был спад
fall_exp = (exp.T.pct_change().iloc[1:] < 0).mean(axis=1) * 100
fall_imp = (imp.T.pct_change().iloc[1:] < 0).mean(axis=1) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for ax, s, title in zip(axes, [fall_exp, fall_imp], ["экспорта", "импорта"]):
    ax.bar(s.index, s.values, color="indianred")
    ax.set_title(f"Доля товаров со снижением {title}, %")
    ax.set_xticks(s.index)
    ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

## 7. Зависимости между товарами

In [ ]:
# корреляция годовых изменений (Спирмен - устойчив к выбросам)
fig, axes = plt.subplots(1, 2, figsize=(26, 10), layout="constrained")
for ax, df, title in zip(axes, [exp, imp], ["Экспорт", "Импорт"]):
    corr = df.T.pct_change().iloc[1:].corr(method="spearman")
    short = [textwrap.shorten(n, 40) for n in corr.index]
    sns.heatmap(corr, annot=True, fmt=".1f", annot_kws={"size": 7}, cmap="coolwarm",
                vmin=-1, vmax=1, xticklabels=short, yticklabels=short, ax=ax)
    ax.set_title(f"{title}: корреляция годовых изменений")
plt.show()

In [ ]:
def top_pairs(df, n=5):
    corr = df.T.pct_change().iloc[1:].corr(method="spearman")
    upper = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool))
    pairs = upper.stack().dropna().sort_values()
    return pd.concat([pairs.head(n), pairs.tail(n)]).round(2)


top_pairs(exp)

In [ ]:
top_pairs(imp)

## 8. Аномалии

In [ ]:
def find_anomalies(df, units, z_limit=3):
    change = df.T.pct_change().iloc[1:] * 100                  # годы x товары
    z = (change - change.mean()) / change.std()
    res = z.stack()
    res = res[res.abs() > z_limit].sort_values(key=abs, ascending=False)
    out = pd.DataFrame({"z": res.round(1)})
    out["изм., %"] = [round(change.loc[y, n]) for y, n in out.index]
    out["единица"] = [units[n] for y, n in out.index]
    out.index.names = ["год", "товар"]
    return out


find_anomalies(exp, exp_units)

In [ ]:
find_anomalies(imp, imp_units)

In [ ]:
# графики товаров с самыми заметными скачками
cases = [
    (exp, exp_units, "Легковые автомобили", "Экспорт"),
    (exp, exp_units, "Сахар", "Экспорт"),
    (imp, imp_units, "Вычислительные машины для автоматической обработки информации", "Импорт"),
    (imp, imp_units, "Машины и механические устройства специального назначения", "Импорт"),
]
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
for ax, (df, units, name, kind) in zip(axes.flat, cases):
    ax.plot(df.columns, df.loc[name], marker="o", ms=3)
    ax.set_title(f"{kind}: {textwrap.fill(name, 45)}")
    ax.set_ylabel(units[name])
plt.tight_layout()
plt.show()